# How Linear Is a Transformer Feed-Forward Block?

Companion notebook for the blog post. Run the cells top to bottom.

In [ ]:
import polyweave
print(polyweave.__version__)

In [ ]:
import torch
import torch.nn as nn
from polyweave.distill import fit_closed_form_linear

# A toy "block": a linear map plus a genuinely nonlinear residual. The (X**2 - 1)
# term is centred, so no linear map can absorb it — it shows up as residual.
torch.manual_seed(0)
d, N = 64, 8_000
X = torch.randn(N, d)
W_true = torch.randn(d, d) / d**0.5
M2 = torch.randn(d, d) / d**0.5
Y = X @ W_true.T + 0.25 * (X**2 - 1.0) @ M2

linear = nn.Linear(d, d)
result = fit_closed_form_linear(linear, X, Y)
print(f"R2_lin = {result.val_r2:.3f}")   # 0.882 — mostly linear, ~12% genuine residual

In [ ]:
import torch, torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from polyweave.distill import IOCapture, fit_closed_form_linear

model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tok = AutoTokenizer.from_pretrained("gpt2")

block = model.transformer.h[1].mlp                      # GPT-2's early FFN sub-block
with IOCapture(block, max_rows=30_000) as cap:
    for text in corpus:                                 # any iterable of strings
        ids = tok(text, return_tensors="pt").input_ids[:, :128]
        with torch.no_grad():
            model(ids)                                  # forward passes drive the hook
X, Y = cap.pairs()                                      # [N, 768], [N, 768]

linear = nn.Linear(X.shape[1], Y.shape[1])
print(fit_closed_form_linear(linear, X, Y).val_r2)      # ≈ 0.95 for GPT-2 block 1